# Instalación completa en un clúster Linux  
## Conda + Enroot + Cobaya + CLASS + PolyChord

Esta guía deja un entorno listo para correr archivos YAML de **Cobaya** con:

- **Cobaya**
- **CLASS / classy**
- **PolyChord / pypolychord**
- **MPI / mpi4py**
- likelihoods de **Planck** y **BICEP/Keck**
- teorías externas en Python como `effective.py`

El flujo diario esperado al final será:

```bash
salloc -p cpu -n 2 -c 1 --mem=16G --time=04:00:00
enroot start 'condaforge+miniforge3' /bin/bash
source ~/enter_inflacion.sh
cd ~/Yaml/test
python -m cobaya run tu_archivo.yaml --resume
```


## 0. Estructura recomendada

Usa esta organización en tu `home`:

```bash
~/miniconda3
~/Yaml
~/cobaya_packages
~/.run/enroot
~/.cache/enroot
~/.local/share/enroot
```

- `~/miniconda3`: instalación de Conda
- `~/Yaml`: YAMLs y archivos Python externos como `effective.py`
- `~/cobaya_packages`: códigos y datos externos de Cobaya
- `.run/.cache/.local/share/enroot`: espacio de trabajo de Enroot


## 1. Instalar Miniconda

In [ ]:
cd ~
wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
bash Miniconda3-latest-Linux-x86_64.sh

# luego
source ~/miniconda3/etc/profile.d/conda.sh
conda --version

## 2. Instalar o configurar Enroot

En muchos clústeres **Enroot ya está instalado** a nivel sistema.  
Prueba primero:

```bash
enroot version
```

Si funciona, no necesitas instalarlo tú.  
Solo debes configurar tus rutas de usuario.


In [ ]:
export ENROOT_RUNTIME_PATH=$HOME/.run/enroot
export ENROOT_CACHE_PATH=$HOME/.cache/enroot
export ENROOT_DATA_PATH=$HOME/.local/share/enroot

mkdir -p "$ENROOT_RUNTIME_PATH" "$ENROOT_CACHE_PATH" "$ENROOT_DATA_PATH"

enroot list

## 3. Crear o usar un contenedor base

In [ ]:
# Si ya existe un contenedor funcional:
enroot start 'condaforge+miniforge3' /bin/bash

Si no existe, el flujo típico es:

1. importar una imagen
2. crear el rootfs
3. arrancar el contenedor


In [ ]:
enroot import docker://condaforge/miniforge3
enroot create --name condaforge+miniforge3 miniforge3*.sqsh
enroot start 'condaforge+miniforge3' /bin/bash

## 4. Crear el entorno Conda

In [ ]:
source ~/miniconda3/etc/profile.d/conda.sh
conda create -n inflacion python=3.10 -y
conda activate inflacion
python -m pip install --upgrade pip

## 5. Instalar Cobaya

In [ ]:
python -m pip install cobaya
python -c "import cobaya; print('cobaya OK')" 

## 6. Instalar compiladores y herramientas de build

In [ ]:
conda activate inflacion
conda install -c conda-forge -y \
  gcc_linux-64 gxx_linux-64 gfortran_linux-64 \
  binutils_linux-64 make cmake openssh

ln -sf "$CONDA_PREFIX/bin/x86_64-conda-linux-gnu-gcc" "$CONDA_PREFIX/bin/gcc"
ln -sf "$CONDA_PREFIX/bin/x86_64-conda-linux-gnu-g++" "$CONDA_PREFIX/bin/g++"
ln -sf "$CONDA_PREFIX/bin/x86_64-conda-linux-gnu-gfortran" "$CONDA_PREFIX/bin/gfortran"
ln -sf "$CONDA_PREFIX/bin/x86_64-conda-linux-gnu-ar" "$CONDA_PREFIX/bin/ar"
ln -sf "$CONDA_PREFIX/bin/x86_64-conda-linux-gnu-ranlib" "$CONDA_PREFIX/bin/ranlib"

which gcc
which gfortran
which ar
which ranlib

## 7. Instalar MPI y mpi4py

In [ ]:
conda activate inflacion
conda install -c conda-forge -y openmpi mpi4py

which mpicc
which mpif90
mpif90 -show

python - <<'PY'
from mpi4py import MPI
print("rank", MPI.COMM_WORLD.Get_rank(), "size", MPI.COMM_WORLD.Get_size())
PY

## 8. Instalar classy

In [ ]:
python -m pip install classy
python -c "import classy; print('classy OK')" 

## 9. Instalar PolyChord

In [ ]:
cd ~/cobaya_packages/code/PolyChordLite

export CC=mpicc
export CXX=mpicxx
export FC=mpif90
export F77=mpif90
export F90=mpif90

make clean
make pypolychord MPI=1
python -m pip install . --no-build-isolation --force-reinstall

cd ~
python -c "import pypolychord; print('pypolychord OK')" 

## 10. Instalar likelihoods y datos externos

In [ ]:
cd ~/Yaml
cobaya-install tu_archivo.yaml --packages-path ~/cobaya_packages --upgrade

## 11. Usar una teoría externa en Python

Guarda `effective.py` en:

```bash
~/Yaml/effective.py
```

Y en el YAML usa un bloque como este:


In [ ]:
theory:
  effective_star:
    python_path: /home/TU_USUARIO/Yaml
    class: effective.EffectiveStarobinsky_SR1

## 12. Crear un script de activación

In [ ]:
cat > ~/enter_inflacion.sh <<'EOF'
source ~/miniconda3/etc/profile.d/conda.sh
conda activate inflacion

export ENROOT_RUNTIME_PATH=$HOME/.run/enroot
export ENROOT_CACHE_PATH=$HOME/.cache/enroot
export ENROOT_DATA_PATH=$HOME/.local/share/enroot

export CC=mpicc
export CXX=mpicxx
export FC=mpif90
export F77=mpif90
export F90=mpif90

export OMPI_MCA_plm_rsh_agent="$CONDA_PREFIX/bin/ssh"
export PYTHONPATH=/home/TU_USUARIO/Yaml:$PYTHONPATH
EOF

source ~/enter_inflacion.sh

## 13. Probar que todo funciona

In [ ]:
cd ~

python - <<'PY'
mods = ["cobaya", "classy", "mpi4py", "pypolychord", "numpy", "scipy"]
for m in mods:
    try:
        __import__(m)
        print(m, "OK")
    except Exception as e:
        print(m, "FAIL", e)
PY

python -c "from effective import EffectiveStarobinsky_SR1; print('effective OK')" 

## 14. Correr un YAML

In [ ]:
cd ~/Yaml/test

# Test de inicialización
python -m cobaya run tu_archivo.yaml --test --force

# Corrida normal
python -m cobaya run tu_archivo.yaml

# Reanudar
python -m cobaya run tu_archivo.yaml --resume

# Empezar de cero
python -m cobaya run tu_archivo.yaml --force

## 15. Flujo diario recomendado

In [ ]:
# desde login
salloc -p cpu -n 2 -c 1 --mem=16G --time=04:00:00

# ya en compute
enroot start 'condaforge+miniforge3' /bin/bash

# dentro del contenedor
source ~/enter_inflacion.sh

cd ~/Yaml/test
python -m cobaya run tu_archivo.yaml --resume

## 16. Qué se conserva y qué no

### Sí se conserva
- `~/miniconda3`
- `~/Yaml`
- `~/cobaya_packages`
- cambios en el código fuente
- contenedores y datos de Enroot

### No queda activo al cerrar sesión
- `conda activate inflacion`
- variables `export`
- estar dentro del contenedor
- sesiones interactivas no protegidas

Por eso es útil `~/enter_inflacion.sh`.


## 17. Errores comunes

### `classy` no instala
Revisa:

```bash
which gcc
which gfortran
which ar
which ranlib
```

### `pypolychord` no compila
Revisa:

```bash
which mpicc
which mpif90
mpif90 -show
```

Y exporta:

```bash
export CC=mpicc
export CXX=mpicxx
export FC=mpif90
export F77=mpif90
export F90=mpif90
```

### Cobaya no encuentra `effective.py`
Usa `python_path` en el YAML o exporta:

```bash
export PYTHONPATH=/home/TU_USUARIO/Yaml:$PYTHONPATH
```

### Cobaya no encuentra likelihoods
Instala dependencias con:

```bash
cobaya-install tu_archivo.yaml --packages-path ~/cobaya_packages --upgrade
```


## 18. Recomendaciones finales

- usa siempre el mismo contenedor
- usa siempre el mismo entorno `inflacion`
- deja `python_path` dentro del YAML
- usa `python -m cobaya run ...`
- usa `--resume` cuando una corrida quede incompleta
- no subas a GitHub `chains/`, `cobaya_packages/`, datos externos ni entornos
